# 03. Build the master dataset

One row per listing, joining the three raw files into
`data/processed/master_listings.parquet`. This is the only dataset the
modelling notebooks will read.

**The master dataset contains no fitted statistic.** No imputed value, no
scaled column, no target encoding, no outlier fence. Every one of those is
learned from the training split, so putting any of them here would leak the
test set into the data before it is even split. What this notebook does is
parse, derive and aggregate, which are all row-local or snapshot-wide
operations that cannot leak.

The one deliberate exception to "leave missing as missing" is a count whose
true value is zero. A listing with no reviews really did receive zero reviews
in the last 90 days; that is the value, not a guess. Dates derived from
reviews that do not exist stay missing. Each case is stated in the data
dictionary.

Other rules held here:

- The calendar price columns are never read. Only the availability flag and
  the date.
- Every window is measured from the snapshot date in `last_scraped`.
- The review text is never loaded.
- Columns that are entirely empty, or constant across the priced rows, are
  excluded rather than carried as dead weight. Each exclusion is documented.

Sections:

0. Setup and the snapshot date
1. Listings: parse and clean
2. Calendar aggregates
3. Review aggregates
4. Assemble and drop unpriced rows
5. Clean the rating columns
6. Scope flag: `long_stay`
7. Duplicate rows
8. Dtypes and column order
9. Save the dataset
10. Column sets
11. `docs/data_dictionary.md`
12. `reports/dataset_build_report.json`
13. Verification, including five hand spot checks

## 0. Setup and the snapshot date

The snapshot date is asserted to be single valued in the listings table, then
used for every window in sections 2 and 3. Nothing anchors to today.

Parquet is now a hard requirement rather than a fallback. The master carries
pandas nullable dtypes (`Int16`, `Int8`, `boolean`, `string`), and a CSV round
trip widens every one of them to `float64` or `object`, which loses the
distinction between a missing integer and a float. Parquet stores the dtype
with the data, so section 13 can assert an exact round trip.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (  # noqa: E402
    NO_REVIEWS_FLAG,
    PROCESSED_DATA_DIR,
    RAW_DATA_DIR,
    REPORTS_DIR,
)
from src.preprocessing.reviews import (  # noqa: E402
    REVIEW_SCORE_COLUMNS,
    flag_no_reviews,
)
from src.preprocessing.target import parse_price  # noqa: E402

DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)

CURRENCY = "AUD"
OCCUPANCY_WINDOWS = (30, 60, 90, 365)
REVIEW_WINDOWS = (90, 180, 365)
RATING_MIN, RATING_MAX = 1.0, 5.0
LONG_STAY_NIGHTS = 28


def resolve_raw(stem: str) -> Path:
    """Return the raw file for a stem, accepting .csv or .csv.gz."""
    for suffix in (".csv", ".csv.gz"):
        candidate = RAW_DATA_DIR / f"{stem}{suffix}"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"no {stem}.csv or {stem}.csv.gz in the raw data folder")


# Row counts are recorded at every step for reports/dataset_build_report.json.
steps: list[dict] = []


def record(step: str, rows: int, note: str = "") -> None:
    steps.append({"step": step, "rows": int(rows), "note": note})
    print(f"  {step:<46} {rows:>9,}  {note}")


print(f"project folder   : {PROJECT_ROOT.name}")
print(f"pyarrow          : {pyarrow.__version__}")
print(f"rating valid range: {RATING_MIN} to {RATING_MAX}")
print(f"long stay cutoff : {LONG_STAY_NIGHTS} nights or more")

project folder   : pricing-recommendation-system
pyarrow          : 18.1.0
rating valid range: 1.0 to 5.0
long stay cutoff : 28 nights or more


## 1. Listings: parse and clean

Three columns need parsing, and in this download two of them have to be
rebuilt from a different source because the obvious column is entirely empty.

- `price` to `price_num` with `parse_price` from `src.preprocessing.target`.
- `bathrooms` is 100% empty, so the count comes from `bathrooms_text`. Rows
  where the text is also missing stay NaN.
- `host_since` is also 100% empty, so tenure comes from the scraper's
  `hosts_time_as_host_years` and `_months`, which are already measured at the
  snapshot date.

`host_id` is carried through as a second identifier. It is **never a feature**.
It is here so a later split can be grouped by host: two listings from the same
host are not independent observations, and a plain random split puts some of a
host's listings in train and the rest in test, which leaks that host's pricing
habits across the boundary. It is also what section 7 uses to decide which
duplicate rows are safe to remove.

`verifications_count`, `instant_bookable` and `host_response_rate` are
included only if the source column has data. Each is checked here and excluded
when empty, so the master never carries an all-NaN column.

In [2]:
LISTING_COLUMNS = [
    "id", "host_id", "last_scraped", "price",
    "accommodates", "bedrooms", "beds", "bathrooms", "bathrooms_text",
    "room_type", "neighbourhood_cleansed", "latitude", "longitude",
    "minimum_nights", "amenities", "host_verifications",
    "host_identity_verified", "host_is_superhost", "instant_bookable",
    "host_response_rate", "host_since",
    "hosts_time_as_host_years", "hosts_time_as_host_months",
    "calculated_host_listings_count",
    "number_of_reviews", "number_of_reviews_ltm", "reviews_per_month",
    *REVIEW_SCORE_COLUMNS,
    "availability_365",
]

print("row counts at each step")
listings = pd.read_csv(resolve_raw("listings"), usecols=LISTING_COLUMNS,
                       low_memory=False)
record("listings.csv rows read", len(listings))

snapshot_values = sorted(listings["last_scraped"].dropna().unique())
assert len(snapshot_values) == 1, f"last_scraped is not single valued: {snapshot_values}"
SNAPSHOT_DATE = pd.Timestamp(snapshot_values[0])
print(f"\nsnapshot date: {SNAPSHOT_DATE.date()}")

assert listings["id"].is_unique, "duplicate listing ids in listings.csv"
print(f"listing ids unique : {listings['id'].is_unique}")
print(f"distinct host ids  : {listings['host_id'].nunique():,} "
      f"across {len(listings):,} listings")
print(f"listings per host  : median "
      f"{listings.groupby('host_id').size().median():.0f}, max "
      f"{listings.groupby('host_id').size().max():,}")


def amenities_count(values: pd.Series) -> pd.Series:
    """Length of the JSON amenity list, NaN when it cannot be parsed."""

    def count(raw):
        if not isinstance(raw, str) or not raw.strip():
            return np.nan
        try:
            parsed = json.loads(raw)
        except (json.JSONDecodeError, ValueError):
            return float(len([p for p in raw.strip("{}[]").split(",") if p.strip()]))
        return float(len(parsed)) if isinstance(parsed, list) else np.nan

    return values.map(count).astype("float64")


def bathrooms_from_text(values: pd.Series) -> pd.Series:
    """
    Bathroom count from the free-text column.

    Handles the three shapes the column takes: a leading number ("1 bath",
    "1.5 shared baths"), a half-bath word form ("Half-bath", "Shared
    half-bath", "Private half-bath") and "No bathroom". Anything else stays
    NaN rather than being guessed at.
    """
    text = values.astype("string").str.strip().str.lower()
    numeric = pd.to_numeric(
        text.str.extract(r"(\d+\.?\d*)", expand=False), errors="coerce"
    ).astype("float64")
    numeric = numeric.mask(text.str.contains("half-bath", na=False), 0.5)
    numeric = numeric.mask(text.eq("no bathroom"), 0.0)
    return numeric.where(text.notna(), np.nan)


def yes_no_to_flag(values: pd.Series) -> pd.Series:
    """Map the t/f convention to 1/0, leaving anything else missing."""
    return values.map({"t": 1.0, "f": 0.0}).astype("float64")


def column_has_data(frame: pd.DataFrame, column: str) -> bool:
    """True when the column exists and is neither all missing nor constant."""
    if column not in frame.columns:
        return False
    values = frame[column]
    return bool(values.notna().any() and values.nunique(dropna=True) > 1)


clean = pd.DataFrame({"id": listings["id"], "host_id": listings["host_id"]})
clean["price_num"] = parse_price(listings["price"])

for column in ("accommodates", "bedrooms", "beds", "minimum_nights",
               "calculated_host_listings_count", "number_of_reviews",
               "number_of_reviews_ltm", "reviews_per_month",
               "availability_365", "latitude", "longitude",
               *REVIEW_SCORE_COLUMNS):
    clean[column] = pd.to_numeric(listings[column], errors="coerce")

clean["bathrooms"] = pd.to_numeric(listings["bathrooms"], errors="coerce")
parsed_bathrooms = bathrooms_from_text(listings["bathrooms_text"])
bathrooms_filled_from_text = int(
    (clean["bathrooms"].isna() & parsed_bathrooms.notna()).sum()
)
clean["bathrooms"] = clean["bathrooms"].fillna(parsed_bathrooms)

clean["room_type"] = listings["room_type"].astype("string")
clean["neighbourhood_cleansed"] = listings["neighbourhood_cleansed"].astype("string")
clean["amenities_count"] = amenities_count(listings["amenities"])
clean["host_identity_verified"] = yes_no_to_flag(listings["host_identity_verified"])
clean["host_is_superhost"] = yes_no_to_flag(listings["host_is_superhost"])
clean["host_tenure_years"] = (
    pd.to_numeric(listings["hosts_time_as_host_years"], errors="coerce")
    + pd.to_numeric(listings["hosts_time_as_host_months"], errors="coerce") / 12.0
)

print(f"\nbathrooms rebuilt from bathrooms_text : {bathrooms_filled_from_text:,} rows")
print(f"bathrooms still missing               : "
      f"{int(clean['bathrooms'].isna().sum()):,} rows")

CONDITIONAL_SOURCES = {
    "verifications_count": "host_verifications",
    "instant_bookable": "instant_bookable",
    "host_response_rate": "host_response_rate",
}
conditional_decisions = {}
print("\nconditional columns, included only if the source has data")
for target, source in CONDITIONAL_SOURCES.items():
    has_data = column_has_data(listings, source)
    conditional_decisions[target] = {
        "source_column": source,
        "included": has_data,
        "pct_missing": round(100 * float(listings[source].isna().mean()), 2)
        if source in listings.columns
        else None,
        "n_unique": int(listings[source].nunique(dropna=True))
        if source in listings.columns
        else None,
    }
    verdict = "INCLUDED" if has_data else "EXCLUDED (empty or constant)"
    print(f"  {target:<22} from {source:<22} "
          f"missing={conditional_decisions[target]['pct_missing']}%  "
          f"n_unique={conditional_decisions[target]['n_unique']}  -> {verdict}")
    if not has_data:
        continue
    if target == "verifications_count":
        clean[target] = amenities_count(listings[source])
    elif target == "instant_bookable":
        clean[target] = yes_no_to_flag(listings[source])
    else:
        clean[target] = pd.to_numeric(
            listings[source].astype(str).str.rstrip("%"), errors="coerce"
        )

clean[NO_REVIEWS_FLAG] = flag_no_reviews(listings).to_numpy()
print(f"\n{NO_REVIEWS_FLAG}: {int(clean[NO_REVIEWS_FLAG].sum()):,} listings "
      f"({clean[NO_REVIEWS_FLAG].mean():.2%})")
record("listings parsed and cleaned", len(clean))

row counts at each step


  listings.csv rows read                            20,833  

snapshot date: 2026-09-17
listing ids unique : True
distinct host ids  : 9,769 across 20,833 listings
listings per host  : median 1, max 230

bathrooms rebuilt from bathrooms_text : 20,809 rows
bathrooms still missing               : 24 rows

conditional columns, included only if the source has data
  verifications_count    from host_verifications     missing=100.0%  n_unique=0  -> EXCLUDED (empty or constant)
  instant_bookable       from instant_bookable       missing=100.0%  n_unique=0  -> EXCLUDED (empty or constant)
  host_response_rate     from host_response_rate     missing=100.0%  n_unique=0  -> EXCLUDED (empty or constant)

has_no_reviews: 3,210 listings (15.41%)
  listings parsed and cleaned                       20,833  


## 2. Calendar aggregates

Only `listing_id`, `date` and `available` are read, with compact dtypes. The
header is asserted to contain no price column before anything is loaded.

`occ_N` is the share of the next N days from the snapshot date that are marked
unavailable. `n_availability_blocks_365` counts separate runs of unavailable
dates, which is what distinguishes one long host-blocked stretch from many
individual bookings. Section 9 of notebook 02 covers why these are upper
bounds on occupancy rather than measurements of it.

`blocked_all_year` is computed here and then **dropped**. The reason is
measured rather than assumed: every listing whose calendar is shut for all 365
days is also a listing with no price, so once the master is restricted to
priced rows the flag is constant and carries no information. The count is
printed so the finding is on the record.

In [3]:
FORBIDDEN_CALENDAR_COLUMNS = {"price", "adjusted_price"}
calendar_header = pd.read_csv(resolve_raw("calendar"), nrows=0).columns.tolist()
print(f"calendar header: {calendar_header}")
print(f"price-like columns in the header: "
      f"{sorted(FORBIDDEN_CALENDAR_COLUMNS & set(calendar_header)) or 'none'}")

calendar = pd.read_csv(
    resolve_raw("calendar"),
    usecols=["listing_id", "date", "available"],
    dtype={"listing_id": "int64", "available": "category"},
    parse_dates=["date"],
)
assert not FORBIDDEN_CALENDAR_COLUMNS & set(calendar.columns), "a price column was read"
record("calendar.csv listing-day rows read", len(calendar))

calendar["unavailable"] = (calendar["available"] == "f").to_numpy()
calendar["day_offset"] = (calendar["date"] - SNAPSHOT_DATE).dt.days.astype("int16")
calendar["is_weekend"] = calendar["date"].dt.dayofweek >= 5
calendar = calendar.drop(columns=["available", "date"])

assert calendar["day_offset"].min() == 0, "calendar starts before the snapshot date"
print(f"\nday_offset range : {calendar['day_offset'].min()} to "
      f"{calendar['day_offset'].max()}")
print(f"unique listings  : {calendar['listing_id'].nunique():,}")

calendar_aggregates = pd.DataFrame(
    index=pd.Index(sorted(calendar["listing_id"].unique()), name="id")
)
for window in OCCUPANCY_WINDOWS:
    calendar_aggregates[f"occ_{window}"] = (
        calendar.loc[calendar["day_offset"] < window]
        .groupby("listing_id")["unavailable"]
        .mean()
    )

calendar_aggregates["weekend_occ_365"] = (
    calendar.loc[calendar["is_weekend"]].groupby("listing_id")["unavailable"].mean()
)
calendar_aggregates["weekday_occ_365"] = (
    calendar.loc[~calendar["is_weekend"]].groupby("listing_id")["unavailable"].mean()
)
calendar_aggregates["weekend_minus_weekday_occ"] = (
    calendar_aggregates["weekend_occ_365"] - calendar_aggregates["weekday_occ_365"]
)

# Runs of unavailable dates, counted on a (listing, day) ordered view.
ordered = calendar.sort_values(["listing_id", "day_offset"], kind="stable")
listing_ids = ordered["listing_id"].to_numpy()
unavailable = ordered["unavailable"].to_numpy()
previous_unavailable = np.concatenate(([False], unavailable[:-1]))
same_listing = np.concatenate(([False], listing_ids[1:] == listing_ids[:-1]))
is_block_start = unavailable & ~(previous_unavailable & same_listing)
calendar_aggregates["n_availability_blocks_365"] = (
    pd.Series(is_block_start, index=ordered["listing_id"]).groupby(level=0).sum()
).astype("float64")

calendar_aggregates["has_calendar_data"] = 1.0

# Measure blocked_all_year, then drop it. See the markdown above.
blocked_all_year = (calendar_aggregates["occ_365"] >= 1.0)
priced_ids = set(clean.loc[clean["price_num"] > 0, "id"])
blocked_and_priced = int(
    blocked_all_year.loc[blocked_all_year.index.isin(priced_ids)].sum()
)
blocked_all_year_decision = {
    "blocked_all_year_all_listings": int(blocked_all_year.sum()),
    "blocked_all_year_priced_listings": blocked_and_priced,
    "dropped": True,
    "reason": "constant across the priced rows, so it carries no information "
              "in the master dataset",
}
print(f"\nblocked_all_year, all {len(calendar_aggregates):,} listings : "
      f"{int(blocked_all_year.sum()):,}")
print(f"blocked_all_year, priced listings only          : {blocked_and_priced:,}")
print("  -> every fully blocked listing is also unpriced, so the flag would be")
print("     constant 0 across the master. DROPPED, and recorded in the data")
print("     dictionary under 'computed then dropped'.")

print(f"\ncalendar aggregates kept for {len(calendar_aggregates):,} listings: "
      f"{list(calendar_aggregates.columns)}")

del calendar, ordered, listing_ids, unavailable
record("listings with calendar aggregates", len(calendar_aggregates))

calendar header: ['listing_id', 'date', 'available', 'minimum_nights', 'maximum_nights']
price-like columns in the header: none


  calendar.csv listing-day rows read             7,604,045  



day_offset range : 0 to 364
unique listings  : 20,833



blocked_all_year, all 20,833 listings : 2,238
blocked_all_year, priced listings only          : 0
  -> every fully blocked listing is also unpriced, so the flag would be
     constant 0 across the master. DROPPED, and recorded in the data
     dictionary under 'computed then dropped'.

calendar aggregates kept for 20,833 listings: ['occ_30', 'occ_60', 'occ_90', 'occ_365', 'weekend_occ_365', 'weekday_occ_365', 'weekend_minus_weekday_occ', 'n_availability_blocks_365', 'has_calendar_data']
  listings with calendar aggregates                 20,833  


## 3. Review aggregates

Only `listing_id` and `date` are read, so the review text never enters the
process. Windows are counted in days before the snapshot date.

Three choices worth stating:

- The count columns are **0** for a listing with no reviews in that window.
  Zero is the true count, not an imputed value.
- `days_since_last_review` and `first_review_age_years` are **missing** for a
  listing with no reviews at all, because there is no date to measure from.
  Filling them would be a guess.
- `review_growth` is `(reviews_last_365d + 1) / (reviews_365_to_730d + 1)`.
  The plain ratio is undefined whenever the earlier year is empty, which is
  common, so each side is smoothed by one. It equals 1 when both years match,
  including when both are zero, and it is left **missing** for a listing with
  no reviews at all, where the ratio would be a meaningless 1.

In [4]:
reviews = pd.read_csv(
    resolve_raw("reviews"),
    usecols=["listing_id", "date"],
    dtype={"listing_id": "int64"},
    parse_dates=["date"],
)
assert set(reviews.columns) == {"listing_id", "date"}, "unexpected review columns read"
record("reviews.csv rows read", len(reviews))

reviews["days_ago"] = (SNAPSHOT_DATE - reviews["date"]).dt.days
print(f"\nreviews dated after the snapshot: {int((reviews['days_ago'] < 0).sum()):,}")
print(f"review date range: {reviews['date'].min().date()} to "
      f"{reviews['date'].max().date()}")

reviewed_ids = pd.Index(sorted(reviews["listing_id"].unique()), name="id")
review_aggregates = pd.DataFrame(index=reviewed_ids)

for window in REVIEW_WINDOWS:
    review_aggregates[f"reviews_last_{window}d"] = (
        reviews.loc[reviews["days_ago"].between(0, window - 1)]
        .groupby("listing_id")
        .size()
    )
prior_year = reviews.loc[reviews["days_ago"].between(365, 729)].groupby("listing_id").size()

# A listing present in reviews.csv but with nothing in a window truly has zero
# there, so these fills are the real counts rather than imputation.
count_columns = [f"reviews_last_{w}d" for w in REVIEW_WINDOWS]
review_aggregates[count_columns] = review_aggregates[count_columns].fillna(0.0)
review_aggregates["reviews_365_to_730d"] = prior_year.reindex(reviewed_ids).fillna(0.0)

review_aggregates["days_since_last_review"] = (
    reviews.groupby("listing_id")["days_ago"].min().astype("float64")
)
review_aggregates["first_review_age_years"] = (
    reviews.groupby("listing_id")["days_ago"].max() / 365.25
)
review_aggregates["review_growth"] = (
    review_aggregates["reviews_last_365d"] + 1.0
) / (review_aggregates["reviews_365_to_730d"] + 1.0)

# reviews_365_to_730d was a working column for the growth ratio only.
review_aggregates = review_aggregates.drop(columns=["reviews_365_to_730d"])

print(f"\nreview aggregates built for {len(review_aggregates):,} listings")
print(review_aggregates.describe(percentiles=[0.5]).round(3).to_string())
record("listings with review aggregates", len(review_aggregates))

  reviews.csv rows read                            829,163  

reviews dated after the snapshot: 0
review date range: 2009-12-05 to 2026-09-17

review aggregates built for 17,623 listings
       reviews_last_90d  reviews_last_180d  reviews_last_365d  days_since_last_review  first_review_age_years  review_growth
count         17623.000          17623.000          17623.000               17623.000               17623.000      17623.000
mean              2.944              5.777             12.584                 239.851                   3.115          4.309
std               4.447              8.184             15.817                 527.606                   3.110          8.504
min               0.000              0.000              0.000                   0.000                   0.000          0.014
50%               1.000              3.000              7.000                  40.000                   2.001          1.333
max             188.000            330.000            529.000  

## 4. Assemble and drop unpriced rows

The join is a left join from the cleaned listings, so the master has exactly
one row per listing before the price filter.

Then the only row-dropping rule based on content: a listing with a missing or
non-positive price cannot serve as a training example for a price model, so it
is dropped and the count is reported. Every other row is kept, extremes
included, because deciding what counts as an outlier is a training-split
decision.

In [5]:
master = (
    clean.merge(calendar_aggregates, left_on="id", right_index=True, how="left")
    .merge(review_aggregates, left_on="id", right_index=True, how="left")
)
record("after joining calendar and reviews", len(master))

master["has_calendar_data"] = master["has_calendar_data"].fillna(0.0)

no_review_rows = master["reviews_last_365d"].isna()
master.loc[no_review_rows, [f"reviews_last_{w}d" for w in REVIEW_WINDOWS]] = 0.0
print(f"listings with no review rows at all : {int(no_review_rows.sum()):,}")
print("  review counts set to 0; days_since_last_review, first_review_age_years")
print("  and review_growth left missing")

missing_price = int(master["price_num"].isna().sum())
non_positive_price = int((master["price_num"] <= 0).sum())
drop_mask = master["price_num"].isna() | (master["price_num"] <= 0)
master = master.loc[~drop_mask].reset_index(drop=True)
print(f"\ndropped for missing price      : {missing_price:,}")
print(f"dropped for non-positive price : {non_positive_price:,}")
record("after dropping unpriced rows", len(master))

  after joining calendar and reviews                20,833  
listings with no review rows at all : 3,210
  review counts set to 0; days_since_last_review, first_review_age_years
  and review_growth left missing

dropped for missing price      : 3,112
dropped for non-positive price : 0
  after dropping unpriced rows                      17,721  


## 5. Clean the rating columns

The seven `review_scores_*` columns are documented as a 1 to 5 scale, but this
download contains values at 0 and values above 5, up to 5.50. A 0 is not a
one-star review and a 5.5 is not a scale point, so both are recording errors
rather than opinions.

Out-of-range values are set to **missing**, not clipped. Clipping would invent
a rating the guests did not give; missing says the rating is unknown, which is
what it is. The count changed per column is reported, and these are the only
missing values this notebook introduces.

This runs **before** the duplicate check in section 7, so two rows that differ
only by an invalid rating are compared on the cleaned values.

In [6]:
rating_changes = {}
print(f"review_scores_* values outside {RATING_MIN} to {RATING_MAX}, set to missing\n")
print(f"{'column':<32} {'below 1':>8} {'above 5':>8} {'changed':>8} "
      f"{'missing before':>15} {'missing after':>14}")
for column in REVIEW_SCORE_COLUMNS:
    values = master[column]
    missing_before = int(values.isna().sum())
    below = int((values < RATING_MIN).sum())
    above = int((values > RATING_MAX).sum())
    invalid = values.notna() & ((values < RATING_MIN) | (values > RATING_MAX))
    master.loc[invalid, column] = np.nan
    missing_after = int(master[column].isna().sum())
    rating_changes[column] = {
        "below_min": below,
        "above_max": above,
        "set_to_missing": int(invalid.sum()),
        "missing_before": missing_before,
        "missing_after": missing_after,
        "observed_min_after": float(master[column].min()),
        "observed_max_after": float(master[column].max()),
    }
    print(f"{column:<32} {below:>8} {above:>8} {int(invalid.sum()):>8} "
          f"{missing_before:>15,} {missing_after:>14,}")

total_ratings_changed = sum(c["set_to_missing"] for c in rating_changes.values())
print(f"\ntotal values set to missing across the seven columns: "
      f"{total_ratings_changed}")
print("\nobserved range after cleaning")
for column, changes in rating_changes.items():
    print(f"  {column:<32} {changes['observed_min_after']:.2f} to "
          f"{changes['observed_max_after']:.2f}")
assert all(
    master[column].dropna().between(RATING_MIN, RATING_MAX).all()
    for column in REVIEW_SCORE_COLUMNS
), "an out-of-range rating survived"
print("\nevery surviving rating is inside the documented range")

review_scores_* values outside 1.0 to 5.0, set to missing

column                            below 1  above 5  changed  missing before  missing after
review_scores_rating                    0        0        0           2,575          2,575
review_scores_accuracy                  5        1        6           2,576          2,582
review_scores_cleanliness               5        1        6           2,576          2,582
review_scores_checkin                   5        1        6           2,576          2,582
review_scores_communication             5        1        6           2,576          2,582
review_scores_location                  5        0        5           2,576          2,581
review_scores_value                     5        2        7           2,576          2,583

total values set to missing across the seven columns: 36

observed range after cleaning
  review_scores_rating             1.00 to 5.00
  review_scores_accuracy           1.00 to 5.00
  review_scores_cleanliness 

## 6. Scope flag: `long_stay`

`long_stay` marks a listing whose minimum stay is 28 nights or more. Twenty
eight nights is the conventional short-stay boundary, and above it the listed
nightly figure is really a monthly rent divided out. That is a different
product from a one-night price, so it may belong in a separate model or be
excluded at modelling time.

The rows are **kept**. The flag records the scope question so it can be
answered later with evidence, rather than quietly deleting a tenth of the
dataset now. It is a scope flag, not a feature: it describes which population
a row belongs to, and `minimum_nights` already carries the same information in
a form a model can use.

It is `boolean` rather than `bool` because `minimum_nights` is missing on a
handful of rows, and a missing minimum stay gives an unknown flag rather than
a false one.

In [7]:
master["long_stay"] = (master["minimum_nights"] >= LONG_STAY_NIGHTS).astype("boolean")
master.loc[master["minimum_nights"].isna(), "long_stay"] = pd.NA

long_stay_count = int(master["long_stay"].sum())
long_stay_unknown = int(master["long_stay"].isna().sum())
print(f"long_stay is True    : {long_stay_count:,} "
      f"({long_stay_count / len(master):.2%})")
print(f"long_stay is False   : "
      f"{int((master['long_stay'] == False).sum()):,}")  # noqa: E712
print(f"long_stay is unknown : {long_stay_unknown:,} "
      f"(minimum_nights missing on these rows)")

print("\nminimum_nights at or above the cutoff, by threshold")
for threshold in (28, 30, 60, 90, 180, 365):
    count = int((master["minimum_nights"] >= threshold).sum())
    print(f"  >= {threshold:4d} nights : {count:>6,} ({count / len(master):6.2%})")

print(f"\nmedian price ({CURRENCY}) by long_stay")
print(
    master.groupby(master["long_stay"].astype("object"), dropna=False)["price_num"]
    .agg(n="size", median="median", mean="mean")
    .round(2)
    .to_string()
)

long_stay is True    : 1,832 (10.34%)
long_stay is False   : 15,877
long_stay is unknown : 12 (minimum_nights missing on these rows)

minimum_nights at or above the cutoff, by threshold
  >=   28 nights :  1,832 (10.34%)
  >=   30 nights :  1,775 (10.02%)
  >=   60 nights :  1,730 ( 9.76%)
  >=   90 nights :  1,712 ( 9.66%)
  >=  180 nights :     35 ( 0.20%)
  >=  365 nights :      4 ( 0.02%)

median price (AUD) by long_stay
               n  median    mean
long_stay                       
False      15877  349.40  508.18
True        1832  204.20  546.11
NaN           12  291.18  282.53


## 7. Duplicate rows

Two rows carrying identical values on every substantive column are either the
same property listed twice, or two genuinely different properties that happen
to match. Which one matters, because a duplicate pair split across the
train/test boundary hands the model a test answer it already saw in training.

The match key is every master column **except** the three that are not
substantive: `id` and `host_id`, which are identifiers, and `long_stay`, which
is a deterministic function of `minimum_nights` and so adds nothing. The key
therefore includes the price, the listing attributes, the exact coordinates and
every calendar and review aggregate.

The removal rule is deliberately narrow. A row is removed only when it matches
another row on that whole key **and** shares its `host_id`, keeping the first.
One host listing the same property twice is a data artefact worth collapsing.
Two *different* hosts with identical rows are not: that could be a genuine
coincidence, or two real units in one building, and deleting one would silently
discard another host's data. Those are reported and left in place.

A looser diagnostic key, price plus listing attributes only, is also reported.
It finds more candidates but is not acted on, because two listings can share
attributes and still be different properties with different histories.

In [8]:
IDENTIFIER_COLUMNS = ["id", "host_id"]
SCOPE_COLUMNS = ["long_stay"]
DUPLICATE_KEY = [
    column
    for column in master.columns
    if column not in IDENTIFIER_COLUMNS + SCOPE_COLUMNS
]
print(f"duplicate key uses {len(DUPLICATE_KEY)} columns "
      f"(every column except {IDENTIFIER_COLUMNS + SCOPE_COLUMNS})\n")

in_any_group = master.duplicated(subset=DUPLICATE_KEY, keep=False)
groups = master.loc[in_any_group].groupby(DUPLICATE_KEY, dropna=False, observed=True)

group_rows = []
for key, members in groups:
    host_ids = members["host_id"].unique()
    group_rows.append(
        {
            "n_rows": len(members),
            "n_hosts": len(host_ids),
            "same_host": len(host_ids) == 1,
            "price_num": float(members["price_num"].iloc[0]),
            "room_type": str(members["room_type"].iloc[0]),
            "neighbourhood": str(members["neighbourhood_cleansed"].iloc[0]),
            "ids": members["id"].tolist(),
            "host_ids": host_ids.tolist(),
        }
    )
duplicate_groups = pd.DataFrame(group_rows)

print(f"rows in some duplicate group : {int(in_any_group.sum()):,}")
print(f"duplicate groups             : {len(duplicate_groups):,}")
if len(duplicate_groups):
    same_host_groups = duplicate_groups.loc[duplicate_groups["same_host"]]
    cross_host_groups = duplicate_groups.loc[~duplicate_groups["same_host"]]
    print(f"  groups sharing one host_id : {len(same_host_groups):,} "
          f"covering {int(same_host_groups['n_rows'].sum()):,} rows")
    print(f"  groups across hosts        : {len(cross_host_groups):,} "
          f"covering {int(cross_host_groups['n_rows'].sum()):,} rows")
    print("\nevery duplicate group")
    print(
        duplicate_groups[["n_rows", "n_hosts", "same_host", "price_num",
                          "room_type", "neighbourhood", "ids", "host_ids"]]
        .to_string(index=False)
    )
else:
    same_host_groups = duplicate_groups
    cross_host_groups = duplicate_groups
    print("  no exact duplicate groups on the full key")

duplicate key uses 42 columns (every column except ['id', 'host_id', 'long_stay'])

rows in some duplicate group : 10
duplicate groups             : 5
  groups sharing one host_id : 5 covering 10 rows
  groups across hosts        : 0 covering 0 rows

every duplicate group
 n_rows  n_hosts  same_host  price_num       room_type neighbourhood                                        ids    host_ids
      2        1       True     140.00    Private room        Sydney [1765733755383666378, 1765744323995642020] [688244302]
      2        1       True     154.29 Entire home/apt      Ashfield [1771787951430805399, 1771796873443077524]  [91961414]
      2        1       True     162.00 Entire home/apt      Ashfield [1772805345887403728, 1772812164664115972]  [91961414]
      2        1       True     162.00 Entire home/apt      Ashfield [1772818673047130007, 1772821159467985862]  [91961414]
      2        1       True     456.00      Hotel room        Sydney [1755080798621191185, 1755080798637403

In [9]:
# Remove only within-host exact duplicates, keeping the first occurrence.
same_host_duplicate = (
    master.duplicated(subset=DUPLICATE_KEY + ["host_id"], keep="first")
)
rows_removed = int(same_host_duplicate.sum())

cross_host_rows = int(
    (in_any_group & ~master.duplicated(subset=DUPLICATE_KEY + ["host_id"], keep=False)).sum()
)

print(f"rows removed (exact duplicate AND same host, keeping first) : {rows_removed:,}")
print(f"rows left in place (exact duplicate, different hosts)       : "
      f"{cross_host_rows:,}")

removed_detail = (
    master.loc[same_host_duplicate, ["id", "host_id", "price_num", "room_type",
                                     "neighbourhood_cleansed"]]
    if rows_removed
    else pd.DataFrame()
)
if rows_removed:
    print("\nrows removed")
    print(removed_detail.to_string(index=False))

master = master.loc[~same_host_duplicate].reset_index(drop=True)
record("after removing same-host duplicates", len(master), "final master row count")

# Diagnostic only. Nothing is removed on this looser key.
DIAGNOSTIC_KEY = [
    "price_num", "accommodates", "bedrooms", "beds", "bathrooms", "room_type",
    "neighbourhood_cleansed", "latitude", "longitude", "minimum_nights",
    "amenities_count",
]
loose_duplicate = master.duplicated(subset=DIAGNOSTIC_KEY, keep=False)
loose_groups = master.loc[loose_duplicate].groupby(DIAGNOSTIC_KEY, dropna=False,
                                                   observed=True)
loose_same_host = sum(
    1 for _, members in loose_groups if members["host_id"].nunique() == 1
)
print(f"\ndiagnostic key (price plus listing attributes and coordinates, "
      f"{len(DIAGNOSTIC_KEY)} columns)")
print(f"  rows matching another row : {int(loose_duplicate.sum()):,}")
print(f"  groups                    : {loose_groups.ngroups:,}")
print(f"  of which single-host      : {loose_same_host:,}")
print("  -> reported only; these rows differ in their calendar or review history,")
print("     so they are not treated as the same listing")

rows removed (exact duplicate AND same host, keeping first) : 5
rows left in place (exact duplicate, different hosts)       : 0

rows removed
                 id   host_id  price_num       room_type neighbourhood_cleansed
1755080798637403998 699580416     456.00      Hotel room                 Sydney
1765744323995642020 688244302     140.00    Private room                 Sydney
1771796873443077524  91961414     154.29 Entire home/apt               Ashfield
1772812164664115972  91961414     162.00 Entire home/apt               Ashfield
1772821159467985862  91961414     162.00 Entire home/apt               Ashfield
  after removing same-host duplicates               17,716  final master row count

diagnostic key (price plus listing attributes and coordinates, 11 columns)
  rows matching another row : 45
  groups                    : 21
  of which single-host      : 19
  -> reported only; these rows differ in their calendar or review history,
     so they are not treated as the same list

## 8. Dtypes and column order

Nullable integer dtypes throughout, so a missing count stays missing instead
of forcing the column to float. Parquet preserves these exactly, which section
13 asserts.

In [10]:
DTYPES = {
    "id": "int64",
    "host_id": "int64",
    "price_num": "float64",
    "accommodates": "Int16",
    "room_type": "string",
    "neighbourhood_cleansed": "string",
    "minimum_nights": "Int32",
    "amenities_count": "Int16",
    "host_identity_verified": "Int8",
    "host_tenure_years": "float64",
    "calculated_host_listings_count": "Int32",
    "bedrooms": "Int16",
    "beds": "Int16",
    "bathrooms": "float64",
    "host_is_superhost": "Int8",
    "number_of_reviews": "Int32",
    "number_of_reviews_ltm": "Int32",
    "reviews_per_month": "float64",
    **{column: "float64" for column in REVIEW_SCORE_COLUMNS},
    "availability_365": "Int16",
    **{f"occ_{w}": "float64" for w in OCCUPANCY_WINDOWS},
    "weekend_occ_365": "float64",
    "weekday_occ_365": "float64",
    "weekend_minus_weekday_occ": "float64",
    "n_availability_blocks_365": "Int16",
    **{f"reviews_last_{w}d": "Int32" for w in REVIEW_WINDOWS},
    "days_since_last_review": "Int32",
    "review_growth": "float64",
    "first_review_age_years": "float64",
    NO_REVIEWS_FLAG: "Int8",
    "has_calendar_data": "Int8",
    "verifications_count": "Int16",
    "instant_bookable": "Int8",
    "host_response_rate": "float64",
    "latitude": "float64",
    "longitude": "float64",
    "long_stay": "boolean",
}

COLUMN_ORDER = [column for column in DTYPES if column in master.columns]
unordered = [column for column in master.columns if column not in DTYPES]
assert not unordered, f"columns with no dtype declared: {unordered}"
master = master[COLUMN_ORDER].astype({c: DTYPES[c] for c in COLUMN_ORDER})

print(f"master shape: {master.shape}")
print("\ndtypes")
print(master.dtypes.astype(str).to_string())
print("\ndtype groups")
print(master.dtypes.astype(str).value_counts().to_string())

master shape: (17716, 45)

dtypes
id                                  int64
host_id                             int64
price_num                         float64
accommodates                        Int16
room_type                          string
neighbourhood_cleansed             string
minimum_nights                      Int32
amenities_count                     Int16
host_identity_verified               Int8
host_tenure_years                 float64
calculated_host_listings_count      Int32
bedrooms                            Int16
beds                                Int16
bathrooms                         float64
host_is_superhost                    Int8
number_of_reviews                   Int32
number_of_reviews_ltm               Int32
reviews_per_month                 float64
review_scores_rating              float64
review_scores_accuracy            float64
review_scores_cleanliness         float64
review_scores_checkin             float64
review_scores_communication       float64


In [11]:
missing_report = pd.DataFrame(
    {
        "dtype": master.dtypes.astype(str),
        "pct_missing": (100 * master.isna().mean()).round(2),
        "n_missing": master.isna().sum(),
        "n_unique": master.nunique(dropna=True),
    }
)
print("missingness and cardinality of the master dataset\n")
print(missing_report.to_string())

constant_columns = missing_report.index[missing_report["n_unique"] <= 1].tolist()
print(f"\ncolumns that are constant over the master rows: "
      f"{constant_columns or 'none'}")
for column in constant_columns:
    print(f"  {column}: single value {master[column].dropna().unique()[0]}")

print("\nno fitted statistic check")
print(f"  rows with any missing value : "
      f"{int(master.isna().any(axis=1).sum()):,} "
      f"({master.isna().any(axis=1).mean():.2%})")
print("  -> missing values are still present, so nothing was imputed")

missingness and cardinality of the master dataset

                                  dtype  pct_missing  n_missing  n_unique
id                                int64         0.00          0     17716
host_id                           int64         0.00          0      8157
price_num                       float64         0.00          0      7916
accommodates                      Int16         0.00          0        16
room_type                        string         0.00          0         4
neighbourhood_cleansed           string         0.00          0        38
minimum_nights                    Int32         0.07         12        57
amenities_count                   Int16         0.00          0        98
host_identity_verified             Int8         0.00          0         2
host_tenure_years               float64         0.01          1       189
calculated_host_listings_count    Int32         0.00          0        68
bedrooms                          Int16         0.12         

  has_calendar_data: single value 1

no fitted statistic check
  rows with any missing value : 2,837 (16.01%)
  -> missing values are still present, so nothing was imputed


## 9. Save the dataset

Parquet only. The previous gzipped CSV master is removed so there is no stale
second copy for a later notebook to read by accident.

In [12]:
OUTPUT_PATH = PROCESSED_DATA_DIR / "master_listings.parquet"
master.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow")
RELATIVE_OUTPUT = f"data/processed/{OUTPUT_PATH.name}"

LEGACY_PATH = PROCESSED_DATA_DIR / "master_listings.csv.gz"
legacy_removed = LEGACY_PATH.exists()
if legacy_removed:
    LEGACY_PATH.unlink()

print(f"written : {RELATIVE_OUTPUT}")
print(f"size    : {OUTPUT_PATH.stat().st_size / 1e6:.2f} MB")
print(f"rows    : {len(master):,}")
print(f"columns : {len(master.columns)}")
print(f"\nlegacy data/processed/{LEGACY_PATH.name} removed: {legacy_removed}")
print(f"files now in data/processed: "
      f"{sorted(p.name for p in PROCESSED_DATA_DIR.iterdir() if p.is_file())}")

written : data/processed/master_listings.parquet
size    : 1.26 MB
rows    : 17,716
columns : 45

legacy data/processed/master_listings.csv.gz removed: True
files now in data/processed: ['.gitkeep', 'master_listings.parquet']


## 10. Column sets

Four sets, and one rule that decides them.

**A column is allowed in a model only if the application can supply it for a
mentor.** That is the whole test. It is not about whether a column correlates
with price; it is about whether the thing exists when a real mentor opens the
pricing screen.

- **core** (8) is what a mentor supplies at registration, or what the system
  derives from what they declare. A model on `core` alone can quote a price to
  a mentor with no history at all, which is the case the platform actually
  needs solved.
- **full** is `core` plus the platform-derived performance, rating,
  availability and demand columns. A new mentor has none of them, so a `full`
  model serves established mentors only. The `core` against `full` comparison
  is the ablation the hypothesis is tested by.
- **space_benchmark** (3) is `bedrooms`, `beds` and `bathrooms`. These are
  physical-property attributes with no honest mentorship equivalent. They are
  strong price predictors on this proxy dataset, so they are kept for one
  ablation that measures how much of the achievable accuracy comes from
  property size alone. They are **not** available to the application and must
  never enter a deployed model.
- **unused** is everything else: the identifiers, the target, the scope flag
  and the raw coordinates.

In [13]:
CORE_COLUMNS = [
    "accommodates",
    "room_type",
    "neighbourhood_cleansed",
    "minimum_nights",
    "amenities_count",
    "host_identity_verified",
    "host_tenure_years",
    "calculated_host_listings_count",
]
FULL_ONLY_COLUMNS = [
    "host_is_superhost",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "reviews_per_month",
    *REVIEW_SCORE_COLUMNS,
    "availability_365",
    *[f"occ_{w}" for w in OCCUPANCY_WINDOWS],
    "weekend_occ_365",
    "weekday_occ_365",
    "weekend_minus_weekday_occ",
    "n_availability_blocks_365",
    *[f"reviews_last_{w}d" for w in REVIEW_WINDOWS],
    "days_since_last_review",
    "review_growth",
    "first_review_age_years",
    NO_REVIEWS_FLAG,
    "has_calendar_data",
]
FULL_COLUMNS = CORE_COLUMNS + FULL_ONLY_COLUMNS
SPACE_BENCHMARK_COLUMNS = ["bedrooms", "beds", "bathrooms"]
UNUSED_COLUMNS = ["latitude", "longitude", "host_id", "id", "price_num", "long_stay"]

ROLES = {
    "id": "identifier",
    "host_id": "identifier",
    "price_num": "target",
    "long_stay": "scope flag",
    "latitude": "not supplied by the app",
    "longitude": "not supplied by the app",
}

COLUMN_SETS = {
    "core": CORE_COLUMNS,
    "full": FULL_COLUMNS,
    "space_benchmark": SPACE_BENCHMARK_COLUMNS,
    "unused": UNUSED_COLUMNS,
}

assert len(CORE_COLUMNS) == 8, f"core must hold 8 columns, got {len(CORE_COLUMNS)}"
assert len(SPACE_BENCHMARK_COLUMNS) == 3, "space_benchmark must hold 3 columns"
assert set(CORE_COLUMNS) <= set(FULL_COLUMNS), "core must be a subset of full"

covered = set(FULL_COLUMNS) | set(SPACE_BENCHMARK_COLUMNS) | set(UNUSED_COLUMNS)
uncovered = [c for c in master.columns if c not in covered]
extra = sorted(covered - set(master.columns))
assert not uncovered, f"columns in no set: {uncovered}"
assert not extra, f"sets name columns that are not in the master: {extra}"

print(f"core            ({len(CORE_COLUMNS):>2}): {CORE_COLUMNS}")
print(f"\nfull            ({len(FULL_COLUMNS):>2}) = core + {len(FULL_ONLY_COLUMNS)}")
print(f"  full only     ({len(FULL_ONLY_COLUMNS):>2}): {FULL_ONLY_COLUMNS}")
print(f"\nspace_benchmark ({len(SPACE_BENCHMARK_COLUMNS):>2}): "
      f"{SPACE_BENCHMARK_COLUMNS}")
print(f"\nunused          ({len(UNUSED_COLUMNS):>2}): {UNUSED_COLUMNS}")
print(f"\nmaster columns  ({len(master.columns):>2}): every one is in exactly one set")

column_sets_payload = {
    "generated_by": "notebooks/03_build_master_dataset.ipynb",
    "dataset": RELATIVE_OUTPUT,
    "snapshot_date": str(SNAPSHOT_DATE.date()),
    "rule": "A column is allowed in a model only if the application can supply "
            "it for a mentor.",
    "sets": COLUMN_SETS,
    "set_sizes": {name: len(columns) for name, columns in COLUMN_SETS.items()},
    "full_only": FULL_ONLY_COLUMNS,
    "roles": ROLES,
    "notes": {
        "core_subset_of_full": True,
        "space_benchmark": "Physical-property attributes with no mentorship "
                           "equivalent. For one ablation only, never for a "
                           "deployed model.",
        "unused": "Identifiers, the target, the scope flag and raw coordinates.",
        "profile_only_fields": "Education, experience and credentials are "
                               "collected for the mentor profile only and are "
                               "not model inputs, because the source dataset "
                               "has no equivalent fields to learn from.",
    },
}

COLUMN_SETS_PATH = REPORTS_DIR / "column_sets.json"
with COLUMN_SETS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(column_sets_payload, handle, indent=2)
print(f"\nwritten : reports/{COLUMN_SETS_PATH.name} "
      f"({COLUMN_SETS_PATH.stat().st_size:,} bytes)")

core            ( 8): ['accommodates', 'room_type', 'neighbourhood_cleansed', 'minimum_nights', 'amenities_count', 'host_identity_verified', 'host_tenure_years', 'calculated_host_listings_count']

full            (36) = core + 28
  full only     (28): ['host_is_superhost', 'number_of_reviews', 'number_of_reviews_ltm', 'reviews_per_month', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communication', 'review_scores_location', 'review_scores_value', 'availability_365', 'occ_30', 'occ_60', 'occ_90', 'occ_365', 'weekend_occ_365', 'weekday_occ_365', 'weekend_minus_weekday_occ', 'n_availability_blocks_365', 'reviews_last_90d', 'reviews_last_180d', 'reviews_last_365d', 'days_since_last_review', 'review_growth', 'first_review_age_years', 'has_no_reviews', 'has_calendar_data']

space_benchmark ( 3): ['bedrooms', 'beds', 'bathrooms']

unused          ( 6): ['latitude', 'longitude', 'host_id', 'id', 'price_num', 'long_stay']

## 11. `docs/data_dictionary.md`

One entry per column: type, definition, source file, missing rate, set, role,
supplier, what a new mentor gets, and a suggested mentor-facing label.

The dictionary also records what was computed and then dropped, what the source
download could not supply, and which profile fields are deliberately not model
inputs.

In [14]:
# column -> (source file, supplied_by, new_mentor_behaviour,
#            mentor-facing wording suggestion, definition)
DICTIONARY = {
    "id": ("listings.csv", "system", "assigned at registration",
           "Mentor service reference",
           "Unique listing identifier. Never a feature; carried for joins and "
           "spot checks."),
    "host_id": ("listings.csv", "system", "assigned at registration",
                "Mentor account reference",
                "Identifier of the account that owns the listing. Never a "
                "feature. Present so a later train/test split can be grouped by "
                "host: two listings from one host are not independent, and a "
                "plain random split would leak that host's pricing habits "
                "across the boundary. Also decides which duplicate rows are "
                "safe to remove."),
    "price_num": ("listings.csv", "mentor", "the value being recommended",
                  f"Current session price ({CURRENCY})",
                  f"Parsed listed price in {CURRENCY} at the snapshot date. The "
                  "prediction target. Extremes are kept; rows with a missing or "
                  "non-positive price are dropped."),
    "accommodates": ("listings.csv", "mentor", "declared at registration",
                     "Maximum participants per session",
                     "Declared capacity of the listing."),
    "room_type": ("listings.csv", "mentor", "declared at registration",
                  "Session format, for example one to one, small group or workshop",
                  "Four-level listing format. Kept instead of the finer property "
                  "type, which nests inside it and would produce collinear "
                  "one-hot blocks."),
    "neighbourhood_cleansed": ("listings.csv", "mentor",
                               "derived from the declared address",
                               "Mentor region or market",
                               "Administrative locality, 38 levels. The locality "
                               "carrier that replaces raw coordinates."),
    "minimum_nights": ("listings.csv", "mentor", "declared policy, commonly 1",
                       "Minimum booking commitment",
                       "Shortest stay the host accepts. Values at 28 or above "
                       "are flagged by long_stay but kept."),
    "amenities_count": ("listings.csv", "mentor", "declared list, may be empty",
                        "Number of skills or services listed",
                        "Length of the declared amenity list. The raw list has "
                        "thousands of distinct values, so it is used as a count."),
    "host_identity_verified": ("listings.csv", "mentor", "0 until verified",
                               "Identity verified",
                               "1 when the platform has verified the account "
                               "holder's identity, 0 otherwise. The "
                               "certification analogue."),
    "host_tenure_years": ("listings.csv", "system", "0",
                          "Years on the platform",
                          "Time as a host in years, from the scraper's year and "
                          "month fields. The join date column is entirely empty "
                          "in this download. Already measured at the snapshot "
                          "date, so no window is anchored to the run date."),
    "calculated_host_listings_count": ("listings.csv", "system", "1",
                                       "Number of services the mentor lists",
                                       "Count of the account's listings in this "
                                       "snapshot. System-derived from what the "
                                       "mentor declares, and defined at "
                                       "registration."),
    "bedrooms": ("listings.csv", "mentor", "no mentorship equivalent",
                 "Not collected, physical-property attribute",
                 "Declared bedroom count. 0 is meaningful for a studio and is "
                 "not a missing value. Benchmark only."),
    "beds": ("listings.csv", "mentor", "no mentorship equivalent",
             "Not collected, physical-property attribute",
             "Declared bed count. Benchmark only."),
    "bathrooms": ("listings.csv", "mentor", "no mentorship equivalent",
                  "Not collected, physical-property attribute",
                  "Bathroom count. The numeric source column is entirely empty "
                  "in this download, so the value is parsed from the free-text "
                  "column; half-baths map to 0.5 and 'no bathroom' to 0. Left "
                  "missing where the text is also missing. Benchmark only."),
    "host_is_superhost": ("listings.csv", "platform", "0, not yet eligible",
                          "Top mentor badge",
                          "1 when the account holds the platform's recognition "
                          "badge. Awarded on ratings and reliability, so it is a "
                          "performance signal, not a profile one."),
    "number_of_reviews": ("listings.csv", "platform", "0",
                          "Total reviews received",
                          "All-time review count. Matches the row count in the "
                          "reviews file exactly in this download."),
    "number_of_reviews_ltm": ("listings.csv", "platform", "0",
                              "Reviews in the last 12 months",
                              "Review count over the trailing twelve months, as "
                              "published in the listings file."),
    "reviews_per_month": ("listings.csv", "platform",
                          "undefined, no review history",
                          "Average reviews per month",
                          "Published review rate. Missing for every listing with "
                          "no reviews, which is structural, not random."),
    "review_scores_rating": ("listings.csv", "platform",
                             "undefined, no review history",
                             "Overall mentee rating",
                             "Overall rating on a 1 to 5 scale. Values outside "
                             "that range were set to missing, not clipped."),
    "review_scores_accuracy": ("listings.csv", "platform",
                               "undefined, no review history",
                               "Rating: the session matched its description",
                               "Accuracy sub-rating, 1 to 5. Out-of-range values "
                               "set to missing."),
    "review_scores_cleanliness": ("listings.csv", "platform",
                                  "undefined, no review history",
                                  "No direct equivalent, quality sub-rating proxy",
                                  "Cleanliness sub-rating, 1 to 5. Out-of-range "
                                  "values set to missing."),
    "review_scores_checkin": ("listings.csv", "platform",
                              "undefined, no review history",
                              "Rating: ease of scheduling and starting",
                              "Check-in sub-rating, 1 to 5. Out-of-range values "
                              "set to missing."),
    "review_scores_communication": ("listings.csv", "platform",
                                    "undefined, no review history",
                                    "Rating: communication",
                                    "Communication sub-rating, 1 to 5. "
                                    "Out-of-range values set to missing."),
    "review_scores_location": ("listings.csv", "platform",
                               "undefined, no review history",
                               "No direct equivalent, convenience proxy",
                               "Location sub-rating, 1 to 5. The most "
                               "price-correlated of the rating columns, because "
                               "it encodes locality."),
    "review_scores_value": ("listings.csv", "platform",
                            "undefined, no review history",
                            "Rating: value for money",
                            "Value sub-rating, 1 to 5. Out-of-range values set "
                            "to missing."),
    "availability_365": ("listings.csv", "platform",
                         "undefined until a calendar is set",
                         "Open slots in the next 12 months",
                         "Published count of available days in the next 365. The "
                         "complement of occ_365 to within rounding."),
    "occ_30": ("calendar.csv", "platform", "undefined, no calendar history",
               "Share of the next 30 days already booked or blocked",
               "Share of the 30 days from the snapshot date marked unavailable. "
               "An upper bound on occupancy: unavailable can mean booked or "
               "host-blocked."),
    "occ_60": ("calendar.csv", "platform", "undefined, no calendar history",
               "Share of the next 60 days already booked or blocked",
               "As occ_30 over 60 days."),
    "occ_90": ("calendar.csv", "platform", "undefined, no calendar history",
               "Share of the next 90 days already booked or blocked",
               "As occ_30 over 90 days."),
    "occ_365": ("calendar.csv", "platform", "undefined, no calendar history",
                "Share of the next 12 months already booked or blocked",
                "As occ_30 over the full 365 days."),
    "weekend_occ_365": ("calendar.csv", "platform",
                        "undefined, no calendar history",
                        "Share of weekend slots already booked or blocked",
                        "Share of Saturdays and Sundays in the next 365 days "
                        "marked unavailable."),
    "weekday_occ_365": ("calendar.csv", "platform",
                        "undefined, no calendar history",
                        "Share of weekday slots already booked or blocked",
                        "Share of Mondays to Fridays in the next 365 days marked "
                        "unavailable."),
    "weekend_minus_weekday_occ": ("calendar.csv", "platform",
                                  "undefined, no calendar history",
                                  "Weekend demand skew",
                                  "weekend_occ_365 minus weekday_occ_365. "
                                  "Positive means weekends fill first, which is "
                                  "the signature of real demand rather than a "
                                  "closed calendar."),
    "n_availability_blocks_365": ("calendar.csv", "platform",
                                  "undefined, no calendar history",
                                  "Number of separate booked or blocked periods",
                                  "Count of separate runs of unavailable dates in "
                                  "the next 365 days. One long run suggests a "
                                  "closed calendar; many short runs suggest "
                                  "individual bookings."),
    "reviews_last_90d": ("reviews.csv", "platform", "0",
                         "Reviews in the last 90 days",
                         "Reviews dated within 90 days before the snapshot. 0 is "
                         "the true count for a listing with none, not an imputed "
                         "value."),
    "reviews_last_180d": ("reviews.csv", "platform", "0",
                          "Reviews in the last 180 days",
                          "As reviews_last_90d over 180 days."),
    "reviews_last_365d": ("reviews.csv", "platform", "0",
                          "Reviews in the last 12 months",
                          "As reviews_last_90d over 365 days. Agrees with the "
                          "published trailing-year count on 98.65% of rows, the "
                          "rest differing by one or two at the window edge."),
    "days_since_last_review": ("reviews.csv", "platform",
                               "undefined, no reviews yet",
                               "Days since the most recent review",
                               "Days from the most recent review to the snapshot "
                               "date. Missing where a listing has no reviews, "
                               "because there is no date to measure from."),
    "review_growth": ("reviews.csv", "platform", "undefined, no reviews yet",
                      "Review volume trend year on year",
                      "(reviews_last_365d + 1) / (reviews in the 365 days before "
                      "that + 1). Each side is smoothed by one so the ratio is "
                      "defined when the earlier year is empty. Above 1 means "
                      "speeding up. Equals 1 when both years match, including "
                      "when both are zero. Missing for listings with no reviews."),
    "first_review_age_years": ("reviews.csv", "platform",
                               "undefined, no reviews yet",
                               "Years since the first review",
                               "Years from the earliest review to the snapshot "
                               "date. Missing where a listing has no reviews."),
    NO_REVIEWS_FLAG: ("derived", "system", "1",
                      "New mentor with no reviews yet",
                      "1 when the listing has no review history. Lets a model "
                      "price 'no track record' as its own state instead of "
                      "inheriting an average rating."),
    "has_calendar_data": ("derived", "system", "0 until a calendar is set",
                          "Calendar connected",
                          "1 when the listing appears in the calendar file. Every "
                          "listing does in this snapshot, so it is constant here "
                          "and cannot contribute. Kept because it will vary on "
                          "platform data, where a mentor may not have connected "
                          "a calendar."),
    "latitude": ("listings.csv", "system", "no mentor equivalent",
                 "Not collected, replaced by a region centroid",
                 "Exact coordinate. Excluded from every model: a mentor does not "
                 "have a latitude, and a tree splitting on it memorises the "
                 "street grid instead of a market effect."),
    "longitude": ("listings.csv", "system", "no mentor equivalent",
                  "Not collected, replaced by a region centroid",
                  "Exact coordinate. Excluded for the same reason as latitude."),
    "long_stay": ("derived", "system", "False for a standard session",
                  "Long engagement rather than a single session",
                  f"True when minimum_nights is {LONG_STAY_NIGHTS} or more. At "
                  "that length the listed nightly figure is really a monthly "
                  "rent divided out, which is a different product. A scope flag, "
                  "not a feature: it says which population a row belongs to, and "
                  "minimum_nights already carries the same information in modelled "
                  "form. Unknown where minimum_nights is missing."),
    "verifications_count": ("listings.csv", "mentor",
                            "0 until the mentor verifies",
                            "Number of completed verifications",
                            "Count of completed account verifications. Included "
                            "only when the source column carries data."),
    "instant_bookable": ("listings.csv", "mentor",
                         "declared policy, defaults to False",
                         "Instant booking enabled",
                         "1 when a guest can book without approval. Included "
                         "only when the source column carries data."),
    "host_response_rate": ("listings.csv", "platform",
                           "undefined until the mentor replies to a request",
                           "Response rate to enquiries",
                           "Share of enquiries the account answers. Included "
                           "only when the source column carries data."),
}

SET_OF_COLUMN = {}
for column in master.columns:
    if column in UNUSED_COLUMNS:
        SET_OF_COLUMN[column] = "unused"
    elif column in SPACE_BENCHMARK_COLUMNS:
        SET_OF_COLUMN[column] = "space_benchmark"
    elif column in CORE_COLUMNS:
        SET_OF_COLUMN[column] = "core"
    else:
        SET_OF_COLUMN[column] = "full"

undocumented = [c for c in master.columns if c not in DICTIONARY]
assert not undocumented, f"undocumented columns: {undocumented}"
print(f"every one of the {len(master.columns)} master columns is documented")

rows = []
for column in master.columns:
    source, supplied_by, new_mentor, wording, definition = DICTIONARY[column]
    rows.append(
        {
            "column": column,
            "type": str(master[column].dtype),
            "definition": definition,
            "source_file": source,
            "pct_missing": round(100 * float(master[column].isna().mean()), 2),
            "set": SET_OF_COLUMN[column],
            "role": ROLES.get(column, "feature"),
            "supplied_by": supplied_by,
            "new_mentor_behaviour": new_mentor,
            "mentor_facing_wording": wording,
        }
    )
dictionary = pd.DataFrame(rows)

print("\ncolumns by set")
print(dictionary["set"].value_counts().to_string())
print("\ncolumns by role")
print(dictionary["role"].value_counts().to_string())
print("\ncolumns by supplier")
print(dictionary["supplied_by"].value_counts().to_string())

every one of the 45 master columns is documented

columns by set
set
full               28
core                8
unused              6
space_benchmark     3

columns by role
role
feature                    39
identifier                  2
not supplied by the app     2
target                      1
scope flag                  1

columns by supplier
supplied_by
platform    26
mentor      10
system       9


In [15]:
HEADER = f"""# Master dataset data dictionary

One entry per column of `{RELATIVE_OUTPUT}`. Generated by
`notebooks/03_build_master_dataset.ipynb`; edit the notebook, not this file.

- **Grain**: one row per listing.
- **Rows**: {len(master):,}
- **Columns**: {len(master.columns)}
- **Snapshot date**: {SNAPSHOT_DATE.date()}. Every time window in this dataset
  is measured relative to this date, never to the date the notebook was run.
- **Currency**: {CURRENCY}. No conversion is applied anywhere in the project.
- **Format**: Parquet, so the nullable dtypes below survive a round trip. A
  CSV round trip widens every `Int*` and `boolean` column, losing the
  difference between a missing integer and a float.
- **Fitted statistics**: none. No imputed value, no scaling, no target
  encoding and no outlier fence is stored here. All of those are learned from
  the training split after the 80/20 division, so that the test set cannot
  influence them. Missing values are left missing.
- **The one exception**: a count whose true value is zero is stored as zero.
  A listing with no reviews received zero reviews in the last 90 days. Dates
  derived from reviews that do not exist are left missing.

## The rule that decides the column sets

**A column is allowed in a model only if the application can supply it for a
mentor.** That is the whole test. It is not about whether a column correlates
with price; it is about whether the thing exists when a real mentor opens the
pricing screen and asks what to charge.

| set | size | meaning |
| --- | --- | --- |
| `core` | {len(CORE_COLUMNS)} | What a mentor supplies at registration, or what the system derives from what they declare. Defined for a brand new mentor, so a `core` model can quote a price to someone with no history. |
| `full` | {len(FULL_COLUMNS)} | `core` plus the platform-derived performance, rating, availability and demand columns. A new mentor has none of them, so a `full` model serves established mentors only. The `core` against `full` comparison is the ablation the hypothesis is tested by. |
| `space_benchmark` | {len(SPACE_BENCHMARK_COLUMNS)} | Physical-property attributes with no honest mentorship equivalent. Strong price predictors on this proxy dataset, so they are kept for one ablation that measures how much achievable accuracy comes from property size alone. **Not available to the application; never in a deployed model.** |
| `unused` | {len(UNUSED_COLUMNS)} | Identifiers, the target, the scope flag and the raw coordinates. |

`core` is a subset of `full`. The machine-readable lists are in
`reports/column_sets.json`.

`role` is one of `feature`, `identifier`, `target`, `scope flag` or
`not supplied by the app`. `supplied_by` is one of `mentor`, `platform` or
`system`. `mentor_facing_wording` suggests the label for the application UI;
where a column is a structural proxy with no honest equivalent on a mentorship
platform, the suggestion says so rather than inventing one.

## Profile fields that are deliberately not model inputs

Education, experience and credentials are collected for the mentor profile
only. They are **not** model inputs, because the source dataset has no
equivalent fields, so there is nothing for a model to learn a price effect
from. Collecting them still serves the product: they are what a mentee reads
when deciding who to book. If the platform later supplies its own priced
records that carry these fields, they become candidates for `core`, since a
mentor declares all three at registration and each has a defined value for a
new mentor.

## Columns

"""

TABLE_COLUMNS = [
    ("column", "column"),
    ("type", "type"),
    ("source_file", "source"),
    ("pct_missing", "missing %"),
    ("set", "set"),
    ("role", "role"),
    ("supplied_by", "supplied by"),
    ("new_mentor_behaviour", "new mentor"),
    ("mentor_facing_wording", "suggested wording"),
    ("definition", "definition"),
]

lines = [HEADER.strip(), ""]
lines.append("| " + " | ".join(label for _, label in TABLE_COLUMNS) + " |")
lines.append("| " + " | ".join("---" for _ in TABLE_COLUMNS) + " |")
for row in dictionary.to_dict(orient="records"):
    cells = []
    for key, _ in TABLE_COLUMNS:
        value = row[key]
        if key == "pct_missing":
            value = f"{value:.2f}"
        cells.append(str(value).replace("|", "\\|"))
    lines.append("| " + " | ".join(cells) + " |")

lines += [
    "",
    "## Column sets in full",
    "",
]
for set_name, set_columns in COLUMN_SETS.items():
    lines.append(f"**{set_name}** ({len(set_columns)}): "
                 + ", ".join(f"`{c}`" for c in set_columns))
    lines.append("")

lines += [
    "## Computed then dropped",
    "",
    "Columns this notebook derives and then removes, with the measurement that",
    "justified removing them. Recorded so the work is not repeated.",
    "",
    "- `blocked_all_year` (share of listings whose calendar is shut for all 365",
    f"  days): {blocked_all_year_decision['blocked_all_year_all_listings']:,} of",
    f"  the {len(calendar_aggregates):,} listings are blocked all year, but only",
    f"  {blocked_all_year_decision['blocked_all_year_priced_listings']:,} of them",
    "  carry a price. Every fully blocked listing is also unpriced, so once the",
    "  master is restricted to priced rows the flag is constant 0 and carries no",
    "  information. The finding itself is useful: a shut calendar marks a dormant",
    "  listing, not a perfectly booked one.",
    "",
    "## Columns that are constant in this snapshot",
    "",
    "A constant column carries no signal and cannot contribute to a model. These",
    "are kept because they will vary on platform data, and documented so the",
    "limitation is not rediscovered during modelling.",
    "",
]
if constant_columns:
    for column in constant_columns:
        lines.append(f"- `{column}`: single value "
                     f"`{master[column].dropna().unique()[0]}`")
else:
    lines.append("- none")

lines += [
    "",
    "## Rating values set to missing",
    "",
    "The seven rating columns are documented as a 1 to 5 scale. This download",
    "contains values at 0 and values above 5. A 0 is not a one-star review and a",
    "5.5 is not a scale point, so both are recording errors. They are set to",
    "missing rather than clipped: clipping would invent a rating the guests did",
    "not give, while missing says the rating is unknown, which is what it is.",
    "",
    "| column | below 1 | above 5 | set to missing | missing after |",
    "| --- | --- | --- | --- | --- |",
]
for column, changes in rating_changes.items():
    lines.append(
        f"| `{column}` | {changes['below_min']} | {changes['above_max']} | "
        f"{changes['set_to_missing']} | {changes['missing_after']:,} |"
    )
lines += [
    "",
    f"Total values set to missing: {total_ratings_changed}. These are the only",
    "missing values this notebook introduces.",
    "",
    "## Duplicate rows removed",
    "",
    "A duplicate pair split across the train/test boundary hands the model a test",
    "answer it already saw in training. The match key is every column except",
    "`id`, `host_id` and `long_stay`, so it includes the price, the listing",
    "attributes, the exact coordinates and every calendar and review aggregate.",
    "",
    "Removal is narrow on purpose. A row is removed only when it matches another",
    "row on that whole key **and** shares its `host_id`, keeping the first. One",
    "account listing the same property twice is an artefact worth collapsing. Two",
    "different accounts with identical rows are not: that could be a genuine",
    "coincidence, or two real units in one building, and deleting one would",
    "silently discard another account's data.",
    "",
    f"- Rows in some duplicate group: {int(in_any_group.sum()):,}",
    f"- Groups sharing one `host_id`: {len(same_host_groups):,}",
    f"- Groups spanning different `host_id` values: {len(cross_host_groups):,}",
    f"- **Rows removed**: {rows_removed:,}",
    f"- Rows left in place because the duplicates are across hosts: "
    f"{cross_host_rows:,}",
    "",
    "## Source columns excluded from this dataset",
    "",
    "Requested columns that this download cannot supply.",
    "",
]
for target, decision in conditional_decisions.items():
    if decision["included"]:
        continue
    lines.append(
        f"- `{target}` from `{decision['source_column']}`: "
        f"{decision['pct_missing']}% missing, "
        f"{decision['n_unique']} distinct values, so it is excluded rather than "
        f"carried as an all-missing column."
    )
lines.append("")

DICTIONARY_PATH = DOCS_DIR / "data_dictionary.md"
text = "\n".join(lines) + "\n"
assert "airbnb" not in text.lower(), "the dictionary must not name the source platform"
DICTIONARY_PATH.write_text(text, encoding="utf-8")
print(f"written : docs/{DICTIONARY_PATH.name}")
print(f"size    : {DICTIONARY_PATH.stat().st_size:,} bytes")
print(f"entries : {len(dictionary)}")
print("the dictionary does not name the source platform")

written : docs/data_dictionary.md
size    : 19,982 bytes
entries : 45
the dictionary does not name the source platform


## 12. `reports/dataset_build_report.json`

Row counts at every step, what was dropped and why, the snapshot date, the
rating cleaning counts, the duplicate decisions and the dtype map.

In [16]:
build_report = {
    "generated_by": "notebooks/03_build_master_dataset.ipynb",
    "data_source": "public short-stay listings snapshot used as a structural proxy",
    "snapshot_date": str(SNAPSHOT_DATE.date()),
    "currency": CURRENCY,
    "grain": "one row per listing",
    "output": {
        "path": RELATIVE_OUTPUT,
        "format": "parquet",
        "engine": f"pyarrow {pyarrow.__version__}",
        "size_bytes": int(OUTPUT_PATH.stat().st_size),
        "rows": int(len(master)),
        "columns": int(len(master.columns)),
        "dtypes": {c: str(t) for c, t in master.dtypes.items()},
        "legacy_csv_removed": bool(legacy_removed),
    },
    "raw_files": {
        stem: {
            "filename": resolve_raw(stem).name,
            "size_bytes": int(resolve_raw(stem).stat().st_size),
        }
        for stem in ("listings", "calendar", "reviews")
    },
    "row_counts_by_step": steps,
    "rows_dropped": [
        {
            "reason": "price missing",
            "rows": missing_price,
            "detail": "a listing with no listed price cannot be a training "
                      "example for a price model",
        },
        {
            "reason": "price not positive",
            "rows": non_positive_price,
            "detail": "none found in this snapshot",
        },
        {
            "reason": "exact duplicate row sharing a host_id",
            "rows": rows_removed,
            "detail": "identical on every column except id, host_id and "
                      "long_stay, and owned by the same host; first kept",
        },
    ],
    "rows_dropped_total": int(missing_price + non_positive_price + rows_removed),
    "duplicates": {
        "key_columns": DUPLICATE_KEY,
        "rows_in_any_group": int(in_any_group.sum()),
        "groups_total": int(len(duplicate_groups)),
        "groups_same_host": int(len(same_host_groups)),
        "groups_cross_host": int(len(cross_host_groups)),
        "rows_removed": rows_removed,
        "rows_kept_cross_host": cross_host_rows,
        "groups": duplicate_groups.to_dict(orient="records")
        if len(duplicate_groups)
        else [],
        "diagnostic_key_columns": DIAGNOSTIC_KEY,
        "diagnostic_rows_matching": int(loose_duplicate.sum()),
        "diagnostic_groups": int(loose_groups.ngroups),
        "diagnostic_groups_single_host": int(loose_same_host),
    },
    "ratings_cleaned": {
        "valid_range": [RATING_MIN, RATING_MAX],
        "policy": "out-of-range values set to missing, not clipped",
        "total_set_to_missing": int(total_ratings_changed),
        "by_column": rating_changes,
    },
    "long_stay": {
        "threshold_nights": LONG_STAY_NIGHTS,
        "rows_true": long_stay_count,
        "rows_false": int(len(master) - long_stay_count - long_stay_unknown),
        "rows_unknown": long_stay_unknown,
        "rows_removed": 0,
        "detail": "scope flag only; every row is kept in the file",
    },
    "blocked_all_year": blocked_all_year_decision,
    "windows": {
        "anchor": "listings.last_scraped",
        "occupancy_days": list(OCCUPANCY_WINDOWS),
        "review_days": list(REVIEW_WINDOWS),
        "review_growth": "(reviews_last_365d + 1) / (reviews in days 365 to 729 "
                         "before the snapshot + 1)",
    },
    "fitted_statistics_stored": [],
    "column_sets": COLUMN_SETS,
    "column_set_sizes": {name: len(c) for name, c in COLUMN_SETS.items()},
    "constant_columns": constant_columns,
    "conditional_columns": conditional_decisions,
    "missingness": {
        column: round(100 * float(master[column].isna().mean()), 2)
        for column in master.columns
    },
}

BUILD_REPORT_PATH = REPORTS_DIR / "dataset_build_report.json"
with BUILD_REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(build_report, handle, indent=2, default=str)

print(f"written : reports/{BUILD_REPORT_PATH.name}")
print(f"size    : {BUILD_REPORT_PATH.stat().st_size:,} bytes")
print("\nrow counts by step")
print(pd.DataFrame(steps).to_string(index=False))

written : reports/dataset_build_report.json
size    : 13,897 bytes

row counts by step
                               step    rows                   note
             listings.csv rows read   20833                       
        listings parsed and cleaned   20833                       
 calendar.csv listing-day rows read 7604045                       
  listings with calendar aggregates   20833                       
              reviews.csv rows read  829163                       
    listings with review aggregates   17623                       
 after joining calendar and reviews   20833                       
       after dropping unpriced rows   17721                       
after removing same-host duplicates   17716 final master row count


## 13. Verification

Structural checks, an exact Parquet round trip, then five hand spot checks.

The spot checks recompute `occ_90` and `reviews_last_365d` for five listings
straight from the raw files, by a different code path from the aggregation in
sections 2 and 3: filter the raw rows for one id, then count. If the two agree
the aggregation is doing what the data dictionary says it does.

The five ids are drawn across the occupancy range rather than at random, so a
fully open listing, a fully blocked one and three in between are all covered.

In [17]:
print("structural checks")

priced_rows = int((parse_price(listings["price"]) > 0).sum())
expected_rows = priced_rows - rows_removed
print(f"  priced listings in the file     : {priced_rows:,}")
print(f"  minus same-host duplicates      : {rows_removed:,}")
print(f"  expected master rows            : {expected_rows:,}")
print(f"  actual master rows              : {len(master):,}")
assert len(master) == expected_rows, "master row count does not reconcile"

assert master["id"].is_unique, "duplicate ids in the master dataset"
print(f"  id is unique                    : {master['id'].is_unique}")
print(f"  distinct host_id values         : {master['host_id'].nunique():,}")
assert master["price_num"].gt(0).all(), "a non-positive price survived the filter"
print("  every price is positive         : True")

object_columns = master.select_dtypes(include="object").columns.tolist()
print(f"  columns left as object dtype    : {object_columns or 'none'}")
assert not object_columns, "object dtype columns should be string or numeric"

assert "blocked_all_year" not in master.columns, "blocked_all_year was not dropped"
print("  blocked_all_year is absent      : True")
print(f"  host_id present as identifier   : "
      f"{'host_id' in master.columns and ROLES['host_id'] == 'identifier'}")
print(f"  long_stay present as scope flag : "
      f"{'long_stay' in master.columns and ROLES['long_stay'] == 'scope flag'}")

print("\nmissing values introduced by this notebook")
introduced = {c: v["set_to_missing"] for c, v in rating_changes.items() if v["set_to_missing"]}
print(f"  invalid ratings set to missing  : {total_ratings_changed} "
      f"across {len(introduced)} columns")
print(f"  long_stay unknown               : {long_stay_unknown} "
      f"(inherited from missing minimum_nights, not introduced by a fill)")
print("  nothing else: no column was imputed, scaled or encoded")

structural checks
  priced listings in the file     : 17,721
  minus same-host duplicates      : 5
  expected master rows            : 17,716
  actual master rows              : 17,716
  id is unique                    : True
  distinct host_id values         : 8,157
  every price is positive         : True
  columns left as object dtype    : none
  blocked_all_year is absent      : True
  host_id present as identifier   : True
  long_stay present as scope flag : True

missing values introduced by this notebook
  invalid ratings set to missing  : 36 across 6 columns
  long_stay unknown               : 12 (inherited from missing minimum_nights, not introduced by a fill)
  nothing else: no column was imputed, scaled or encoded


In [18]:
reloaded = pd.read_parquet(OUTPUT_PATH, engine="pyarrow")
print("parquet round trip")
print(f"  shape      : {reloaded.shape} vs {master.shape}")
assert reloaded.shape == master.shape, "round trip changed the shape"
assert list(reloaded.columns) == list(master.columns), "round trip reordered columns"
print(f"  dtypes identical : {reloaded.dtypes.equals(master.dtypes)}")
assert reloaded.dtypes.equals(master.dtypes), "round trip changed a dtype"
pd.testing.assert_frame_equal(master, reloaded)
print("  every value identical : True  (pandas assert_frame_equal passed)")
print("\n  dtype map as stored")
print(reloaded.dtypes.astype(str).value_counts().to_string())

parquet round trip
  shape      : (17716, 45) vs (17716, 45)
  dtypes identical : True


  every value identical : True  (pandas assert_frame_equal passed)

  dtype map as stored
float64    22
Int32       8
Int16       6
Int8        4
int64       2
string      2
boolean     1


In [19]:
print("five hand spot checks, recomputed from the raw files\n")

sorted_by_occupancy = master.sort_values("occ_90", kind="stable").reset_index(drop=True)
positions = np.linspace(0, len(sorted_by_occupancy) - 1, 5).round().astype(int)
spot_ids = sorted_by_occupancy.loc[positions, "id"].tolist()
print(f"ids chosen at occ_90 positions {positions.tolist()} of "
      f"{len(sorted_by_occupancy):,}: {spot_ids}\n")

raw_calendar = pd.read_csv(
    resolve_raw("calendar"),
    usecols=["listing_id", "date", "available"],
    dtype={"listing_id": "int64", "available": "category"},
    parse_dates=["date"],
)
raw_calendar = raw_calendar.loc[raw_calendar["listing_id"].isin(spot_ids)].copy()

raw_reviews = pd.read_csv(
    resolve_raw("reviews"),
    usecols=["listing_id", "date"],
    dtype={"listing_id": "int64"},
    parse_dates=["date"],
)
raw_reviews = raw_reviews.loc[raw_reviews["listing_id"].isin(spot_ids)].copy()

window_start = SNAPSHOT_DATE - pd.Timedelta(days=364)
spot_rows = []
for listing_id in spot_ids:
    days = raw_calendar.loc[raw_calendar["listing_id"] == listing_id]
    first_90 = days.loc[days["date"] < SNAPSHOT_DATE + pd.Timedelta(days=90)]
    hand_occ_90 = float((first_90["available"] == "f").sum()) / len(first_90)

    listing_reviews = raw_reviews.loc[raw_reviews["listing_id"] == listing_id]
    hand_reviews_365 = int(
        listing_reviews["date"].between(window_start, SNAPSHOT_DATE).sum()
    )

    stored = master.loc[master["id"] == listing_id].iloc[0]
    spot_rows.append(
        {
            "id": listing_id,
            "cal_days_in_90": len(first_90),
            "hand_occ_90": round(hand_occ_90, 6),
            "master_occ_90": round(float(stored["occ_90"]), 6),
            "occ_90_match": bool(np.isclose(hand_occ_90, float(stored["occ_90"]))),
            "hand_reviews_365d": hand_reviews_365,
            "master_reviews_365d": int(stored["reviews_last_365d"]),
            "reviews_match": hand_reviews_365 == int(stored["reviews_last_365d"]),
        }
    )

spot_checks = pd.DataFrame(spot_rows)
print(spot_checks.to_string(index=False))

all_matched = bool(
    spot_checks["occ_90_match"].all() and spot_checks["reviews_match"].all()
)
print(f"\nall five spot checks matched on both columns: {all_matched}")
assert all_matched, "a spot check did not match the master dataset"

build_report["spot_checks"] = spot_checks.to_dict(orient="records")
build_report["spot_checks_all_matched"] = all_matched
with BUILD_REPORT_PATH.open("w", encoding="utf-8") as handle:
    json.dump(build_report, handle, indent=2, default=str)
print(f"\nspot checks appended to reports/{BUILD_REPORT_PATH.name}")

five hand spot checks, recomputed from the raw files

ids chosen at occ_90 positions [0, 4429, 8858, 13286, 17715] of 17,716: [118658, 1740379765117458527, 1725137170358173827, 1733104582845930225, 1731583445337438669]



                 id  cal_days_in_90  hand_occ_90  master_occ_90  occ_90_match  hand_reviews_365d  master_reviews_365d  reviews_match
             118658              90     0.000000       0.000000          True                  0                    0           True
1740379765117458527              90     0.133333       0.133333          True                  1                    1           True
1725137170358173827              90     0.377778       0.377778          True                  0                    0           True
1733104582845930225              90     0.700000       0.700000          True                  0                    0           True
1731583445337438669              90     1.000000       1.000000          True                  0                    0           True

all five spot checks matched on both columns: True

spot checks appended to reports/dataset_build_report.json


In [20]:
print("final state\n")
print(f"  {RELATIVE_OUTPUT}")
print(f"    {master.shape[0]:,} rows x {master.shape[1]} columns, "
      f"{OUTPUT_PATH.stat().st_size / 1e6:.2f} MB parquet")
print(f"  docs/data_dictionary.md          : {len(dictionary)} documented columns")
print(f"  reports/column_sets.json         : "
      f"{COLUMN_SETS_PATH.stat().st_size:,} bytes")
print(f"  reports/dataset_build_report.json: "
      f"{BUILD_REPORT_PATH.stat().st_size:,} bytes")

print("\ncolumn sets")
for set_name, set_columns in COLUMN_SETS.items():
    print(f"\n  {set_name} ({len(set_columns)})")
    for column in set_columns:
        role = ROLES.get(column, "feature")
        suffix = "" if role == "feature" else f"  [{role}]"
        print(f"    {column}{suffix}")

final state

  data/processed/master_listings.parquet
    17,716 rows x 45 columns, 1.26 MB parquet
  docs/data_dictionary.md          : 45 documented columns
  reports/column_sets.json         : 3,348 bytes
  reports/dataset_build_report.json: 15,245 bytes

column sets

  core (8)
    accommodates
    room_type
    neighbourhood_cleansed
    minimum_nights
    amenities_count
    host_identity_verified
    host_tenure_years
    calculated_host_listings_count

  full (36)
    accommodates
    room_type
    neighbourhood_cleansed
    minimum_nights
    amenities_count
    host_identity_verified
    host_tenure_years
    calculated_host_listings_count
    host_is_superhost
    number_of_reviews
    number_of_reviews_ltm
    reviews_per_month
    review_scores_rating
    review_scores_accuracy
    review_scores_cleanliness
    review_scores_checkin
    review_scores_communication
    review_scores_location
    review_scores_value
    availability_365
    occ_30
    occ_60
    occ_90
    o